In [1]:
import numpy as np
import scipy.sparse as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import tenpy
import tenpy.linalg.np_conserved as npc
from tenpy import version as v
print(v.version_summary)

from tenpy.networks.site import BosonSite, SpinSite, GroupedSite, set_common_charges
from tenpy.models.lattice import Lattice, IrregularLattice
from tenpy.models.model import CouplingModel
import tenpy.models.lattice as lattice

from tenpy.networks.mps import MPS
from tenpy.algorithms import tebd, dmrg, tdvp

from tenpy.algorithms.mpo_evolution import ExpMPOEvolution
from tenpy.algorithms.exact_diag import ExactDiag
from tenpy.algorithms.dmrg import TwoSiteDMRGEngine
from tenpy.algorithms.tebd import TEBDEngine


import time
import os
import json
from matplotlib import cm
import uuid

from SpinBosonEnv.GeneralSpinBosonEnv import GeneralSpinBosonEnv
from SpinBosonEnv.CavityArrayAtom import CavityArrayAtom

tenpy 1.1.0 (compiled without HAVE_MKL),
git revision f6a04f86f7aec403098d8a6bf6c01ee4a1222c8d using
python 3.12.13 (main, May 10 2026, 19:30:01) [Clang 22.1.3 ]
numpy 2.5.0, scipy 1.18.0


In [2]:
with open("/home/ihuarte/Escritorio/Ivan/MPS/config.json", "r") as f:
    config = json.load(f)

SB_params = config["SB_params"]
model_params = config["model_params"]
DMRG_options = config["DMRG_options"]

for k, v in SB_params.items():
    SB_params[k] = np.pi if v == "pi" else v


# Create simulation folder
sim_uuid = str(uuid.uuid4())[:8]
write_folder = (
    f"/home/ihuarte/Escritorio/Ivan/MPS/Results/{SB_params["ohmic_model"]}/{sim_uuid}/"
)

w0 = SB_params["w0"]
wc = SB_params["wc"]
Nk = SB_params["Nk"]

w_min = w0 * wc / np.sqrt(w0**2 + 4 * wc**2)

delta = 0.15

g = 0.4


SB_params["delta"] = delta
SB_params["g"] = g

print("**********************************************************")
print(f"w0: {SB_params["w0"]}  wc: {wc}  Nk: {SB_params["Nk"]} g: {g}")
print(f"delta: {SB_params["delta"]} Boson_dim: {model_params["N_max"]}")
print("**********************************************************")

# Spin Boson model init
env_SB = GeneralSpinBosonEnv(SB_params)
print(f"Hk: {env_SB.Hk.shape}")
print(f"Hmap: {env_SB.Hmap.shape}")

# Tenpy init
model_params["L"] = len(env_SB.wlist)
model_params["w"] = env_SB.wlist
model_params["delta"] = SB_params["delta"]
model_params["g"] = env_SB.g0
model_params["J"] = env_SB.Jlist

caa = CavityArrayAtom(model_params, DMRG_options)

""" GROUND STATE AND ENERGY """

initial_state = caa.InitialState(config=[], GS=False)

**********************************************************
w0: 1.0  wc: 20  Nk: 10 g: 0.4
delta: 0.15 Boson_dim: 10
**********************************************************
g0 0.3999999999999999
Hk: (11, 11)
Hmap: (6, 6)


/home/ihuarte/Escritorio/Ivan/MPS/.venv/lib/python3.12/site-packages/tenpy/networks/mps.py:1629: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites, bc, unit_cell_width)


In [12]:

# |psi_down> = (|spin_down> , |0>, |0>, ...., |0>) 
# |psi_up> = (|spin_up> , |0>, |0>, ...., |0>) 
psi_down = MPS.from_product_state(
    caa.lat.mps_sites(),
    ['down'] + ["vac" for _ in range(caa.L)],
    bc='finite'
)
psi_up = MPS.from_product_state(
    caa.lat.mps_sites(),
    ['up'] + ["vac" for _ in range(caa.L)],
    bc='finite'
)

# Paridad de los estados iniciales

Ps = psi_up.expectation_value(["parity"], caa.atpos_idx)
Pb = psi_up.expectation_value(["parity"], caa.bs_idx)
P_global = np.concatenate([Ps, Pb], axis=-1).prod()
print(f"Paridad de up: {P_global}")

Ps = psi_down.expectation_value(["parity"], caa.atpos_idx)
Pb = psi_down.expectation_value(["parity"], caa.bs_idx)
P_global = np.concatenate([Ps, Pb], axis=-1).prod()
print(f"Paridad de down: {P_global}")

# Hacemos DMRG de ambos
eng = TwoSiteDMRGEngine(psi_up, caa, config["DMRG_options"])
E_gr_up, psi_gr_up = eng.run()

eng = TwoSiteDMRGEngine(psi_down, caa, config["DMRG_options"])
E_gr_down, psi_gr_down = eng.run()

print(f"Energía que llega psi_up: {E_gr_up}")
print(f"Energía que llega psi_down: {E_gr_down}")


# Comprobamos que preserva la simetria

Ps = psi_gr_up.expectation_value(["parity"], caa.atpos_idx)
Pb = psi_gr_up.expectation_value(["parity"], caa.bs_idx)
P_global = np.concatenate([Ps, Pb], axis=-1).prod()
print(f"Paridad de ground state up: {P_global}")

Ps = psi_gr_down.expectation_value(["parity"], caa.atpos_idx)
Pb = psi_gr_down.expectation_value(["parity"], caa.bs_idx)
P_global = np.concatenate([Ps, Pb], axis=-1).prod()
print(f"Paridad de ground state down: {P_global}")

/home/ihuarte/Escritorio/Ivan/MPS/.venv/lib/python3.12/site-packages/tenpy/networks/mps.py:1629: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites, bc, unit_cell_width)


Paridad de up: -1.0
Paridad de down: 1.0
Energía que llega psi_up: -0.4759699999787091
Energía que llega psi_down: -0.6046918413487956
Paridad de ground state up: -0.7210335257457336
Paridad de ground state down: 0.7507555907553615


In [4]:
eng = TwoSiteDMRGEngine(psi_down, caa, config["DMRG_options"])
E_gr_down, psi_gr_down = eng.run()

In [5]:
eng = TwoSiteDMRGEngine(psi_up, caa, config["DMRG_options"])
E_gr_up, psi_gr_up = eng.run()

In [6]:
caa.sp.state_labels

{'-0.5': 0, '0.5': 1, 'down': 0, 'up': 1}

In [6]:
E_gr_up, E_gr_down

(np.float64(-0.4759699999787091), np.float64(-0.6046918413487956))

In [10]:
Ps = psi_gr_up.expectation_value(["parity"], caa.atpos_idx)
Pb = psi_gr_up.expectation_value(["parity"], caa.bs_idx)
P_global = np.concatenate([Ps, Pb], axis=-1).prod()
print(f"Paridad de up: {P_global}")

Ps = psi_gr_down.expectation_value(["parity"], caa.atpos_idx)
Pb = psi_gr_down.expectation_value(["parity"], caa.bs_idx)
P_global = np.concatenate([Ps, Pb], axis=-1).prod()
print(f"Paridad de down: {P_global}")

Paridad de up: -0.7210335257457336
Paridad de down: 0.7507555907553612


In [ ]:
Ps = psi_gr_down.expectation_value(["parity"], caa.atpos_idx)
Pb = psi_gr_down.expectation_value(["parity"], caa.bs_idx)
np.concatenate([Ps, Pb], axis=-1).prod()

np.float64(-0.8894453180510441)

In [14]:
from tenpy.networks.mpo import MPO
import inspect

print([m for m in dir(MPO) if "from" in m])

['from_grids', 'from_hdf5', 'from_wavepacket']


In [16]:
print(hasattr(psi_up, "expectation_value_term"))

True


In [23]:
ops = ["parity"] * (psi_up.L)
sites = list(range(psi_up.L))
terms = [(op, i) for op, i in zip(ops, sites)]
parity = psi_up.expectation_value_term(terms)
print(parity)

-1.0000000000000007


In [ ]:
psi_up